# 12 Export: Charts 12–14: what mattered and where they are now

**Owner:** Devarshi

Writes one upload-ready CSV per chart to `outputs/charts/`, its spec entry to `outputs/charts/chart_specs.md`, and the numbers its title uses to `outputs/findings.json` (`export_chart_NN`). Chart tables are built by `src/exports.py` from `data/clean/analysis.parquet` and the notebook 05 findings. Titles are assembled from those numbers, never typed by hand.

Run notebook 05 first. Featured five: `findings.json → featured_five`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import config, exports as ex, findings as fd

pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
NB = "12_export_act4_5"
P = config.PALETTE
ctx = ex.load_ctx()
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append({"Check": name, "Result": "PASS" if passed else "FAIL", "Detail": detail})
    print(("PASS" if passed else "FAIL"), "-", name, ("| " + detail if detail else ""))

def publish(n, short, df, facts, spec):
    check(f"Chart {n:02d}: under {config.CHART_MAX_ROWS} rows", len(df) <= config.CHART_MAX_ROWS, f"{len(df)} rows")
    ex.save_csv(n, short, df)
    fd.update(f"export_chart_{n:02d}", facts, NB)
    ex.write_spec(n, spec)
    print(f"wrote outputs/charts/{n:02d}_{short}.csv ({len(df)} rows) and its spec")
    display(df.head(8))
print("Featured five:", list(ctx.featured_names.values()))

Featured five: ['R. Sterling', 'R. Živković', 'H. Kane', 'M. Depay', 'T. Werner']


## Chart 12: who_predicted_best

In [2]:
df, f = ex.chart_12(ctx)
spec = ex.spec_entry(12, "who_predicted_best", "Datawrapper range plot (AUC with 95% CI)",
    "The game's potential predicted who would make it as well as the transfer market did",
    f"AUC {f['potential_auc']:.2f} for EA potential vs {f['market_auc']:.2f} for market value at entry. AUC: the chance a predictor ranks a player who made it (3,000+ Big-5 minutes) above one who didn't; 0.5 = coin flip, 1 = perfect. {f['n']:,} matched entrants with a market value at entry",
    "rows = Predictor; range start = CI low; range end = CI high; dot = AUC: who made it; tooltip adds 'Correlation with peak later rating' and 'What it is'",
    f"EA potential {P['promise']}; market value {P['reality']}; the two benchmarks {P['context']}",
    "<b>{{ Predictor }}</b><br>AUC: {{ AUC: who made it }} (95% CI {{ CI low }}–{{ CI high }})<br>Correlation with the game's own later rating: {{ Correlation with peak later rating }}<br>{{ What it is }}",
    f"x axis 0.5–0.8. Potential vs market: difference {f['potential_minus_market']['value']} (95% CI {f['potential_minus_market']['ci_low']} to {f['potential_minus_market']['ci_high']}), not distinguishable. Potential still adds information beyond market value (AUC {f['potential_within_value_quintiles']['value']} within value quintiles). The game's own current rating predicts its future rating better than potential does ({f['spearman_overall']} vs {f['spearman_potential']}): put that in the text or the notes.")
publish(12, "who_predicted_best", df, f, spec)

PASS - Chart 12: under 3000 rows | 4 rows
wrote outputs/charts/12_who_predicted_best.csv (4 rows) and its spec


,Predictor,AUC: who made it,CI low,CI high,Correlation with peak later rating,What it is
0,EA potential rating,0.714,0.696,0.732,0.50,The game's promise for the player
1,Transfermarkt market value at entry,0.716,0.699,0.733,NaN,What the transfer market paid for
2,Simple formula: overall + age,0.701,0.682,0.718,0.56,Out-of-sample fit on the other editions
3,EA overall rating (current ability),0.692,0.674,0.711,0.57,"The game's rating today, no promise"


## Chart 13: game_vs_market

In [3]:
panel = pd.read_parquet(config.CLEAN / "fifa_panel.parquet", columns=["player_id", "edition", "overall", "edition_date"])
val = pd.read_csv(config.RAW_TM / "player_valuations.csv", parse_dates=["date"])
df, f = ex.chart_13(ctx, panel, val)
spec = ex.spec_entry(13, "game_vs_market", "Flourish line chart with player filter",
    "Two verdicts on the same careers: the game's rating and the market's price",
    "Each player's game rating and market value as a % of his own peak, 2014–2026. Featured five plus the ten most valuable other entrants",
    "x = Date; lines = 'Game rating (% of own peak)' and 'Market value (% of own peak)'; filter = Player (default: first featured player)",
    f"game rating {P['promise']}; market value {P['reality']}",
    "<b>{{ Player }}</b> · {{ Date }}<br>{{ series }}: {{ value }}% of his peak",
    "Rows have a value in one series or the other (game ratings at edition dates, valuations at Transfermarkt dates): turn ON 'Connect gaps' / interpolation. y axis 0–100%. Set the filter to a dropdown; Featured players first.")
publish(13, "game_vs_market", df, f, spec)

PASS - Chart 13: under 3000 rows | 553 rows
wrote outputs/charts/13_game_vs_market.csv (553 rows) and its spec


,Date,Player,Game rating (% of own peak),Market value (% of own peak),Featured
387,2016-06-17,A. Isak,<NA>,0,
388,2016-12-11,A. Isak,<NA>,1,
389,2017-02-07,A. Isak,<NA>,4,
390,2017-06-19,A. Isak,<NA>,4,
386,2017-09-18,A. Isak,83,<NA>,
391,2017-12-28,A. Isak,<NA>,4,
392,2018-06-05,A. Isak,<NA>,4,
385,2018-08-21,A. Isak,84,<NA>,


## Chart 14: where_they_ended_up

In [4]:
df, f = ex.chart_14(ctx)
t = f["tier_pct"]
spec = ex.spec_entry(14, "where_they_ended_up", "Flourish Sankey",
    f"Of {f['n']:,} prospects rated potential 75+, {t['Big-5']}% play in the Big-5 today",
    "Potential band at entry → status in the game → latest club (Transfermarkt, 2025/26 season)",
    "Source = Source; Target = Target; Value = Value; Step column optional (Flourish infers steps from the node names)",
    f"bands {P['promise']} shades; statuses Reached {P['reality']}, Fell short {P['fell_short']}, Vanished {P['context']}; tiers: Big-5 {P['beat']}, others in greys",
    "<b>{{ source }} → {{ target }}</b><br>{{ value }} players",
    "Latest-club tier rule is provisional (src/outcomes.py, DECISIONS.md): moves Transfermarkt never recorded (e.g. to Qatar) count as 'Other or lower league'. 'Not in Transfermarkt' = never matched (mostly never reached a covered league).")
publish(14, "where_they_ended_up", df, f, spec)

PASS - Chart 14: under 3000 rows | 26 rows
wrote outputs/charts/14_where_they_ended_up.csv (26 rows) and its spec


,Source,Target,Value,Step
0,Potential 75–79,Game: fell short,1343,1: band to game status
1,Potential 75–79,Game: reached,290,1: band to game status
2,Potential 75–79,Game: vanished,151,1: band to game status
3,Potential 80–84,Game: fell short,487,1: band to game status
4,Potential 80–84,Game: reached,101,1: band to game status
5,Potential 80–84,Game: vanished,12,1: band to game status
6,Potential 85+,Game: fell short,89,1: band to game status
7,Potential 85+,Game: reached,11,1: band to game status


## Summary

In [5]:
checks = pd.DataFrame(CHECKS); display(checks)
assert (checks["Result"] == "PASS").all()

,Check,Result,Detail
0,Chart 12: under 3000 rows,PASS,4 rows
1,Chart 13: under 3000 rows,PASS,553 rows
2,Chart 14: under 3000 rows,PASS,26 rows
